# Voice anti-spoofing v3 (GPU training)

This notebook trains the **v3** pipeline on Colab. It is designed to run on a **T4 GPU** and to keep checkpoints in Google Drive so a disconnected session can be resumed.

What v3 changes compared with v1/v2:

- **Layer fusion:** softmax-weighted combination of all 25 XLS-R hidden states (v1 used only the last layer).
- **More diverse data:** ASVspoof 2019 LA + MLAAD-tiny, balanced per `(source, label)`.
- **Shortcut control:** RMS normalisation to -23 dB, random noise-floor equalisation, noise/reverb/codec augmentation, RawBoost algos 4/5/8.
- **System-disjoint splits:** held-out MLAAD TTS systems for dev and a completely unseen system set for test.
- **Focal loss** and checkpoint selection by **mean AUC over ASV dev + MLAAD dev** (not ASV dev alone).
- **Calibrated demo** with a temperature and a two-threshold (real / uncertain / fake) decision.

**Before running:** upload these three files to your My Drive root (`MyDrive/`, not a subfolder):

1. `colab_pack.zip` (ASVspoof + MLAAD all/matched pack)
2. `colab_pack_v2.zip` (MLAAD train2 + unseen-system test)
3. `colab_v3_code.zip` (run `python3 pack_v3.py` on your Mac to create it)

Run the cells top to bottom. The first code cell mounts Drive safely; if a previous failed mount left files in `/content/drive`, it moves them to `/content/drive_stale_...` instead of failing. The next cell checks the zips and creates the output folder.

Runtime -> Change runtime type -> **T4 GPU**. A 12-epoch run is roughly 2-4 hours on a free T4. If you get an out-of-memory error, set `BATCH = 4` and rerun; the script has no gradient accumulation, so a smaller batch just means more steps.

In [ ]:
import os
import subprocess
import time

from google.colab import drive

MOUNT = '/content/drive'


def _is_mountpoint(path):
    try:
        return subprocess.run(
            ['mountpoint', '-q', path], capture_output=True
        ).returncode == 0
    except FileNotFoundError:
        return os.path.ismount(path)


if os.path.islink(MOUNT):
    os.remove(MOUNT)

if os.path.exists(MOUNT) and not os.path.isdir(MOUNT):
    os.remove(MOUNT)

if os.path.isdir(MOUNT) and os.listdir(MOUNT) and not _is_mountpoint(MOUNT):
    stale = f'/content/drive_stale_{int(time.time())}'
    os.rename(MOUNT, stale)
    print('moved stale', MOUNT, '->', stale)
    print('inspect it, then remove with:  !rm -rf', stale)

drive.mount(MOUNT, force_remount=True)

## 1. Set paths, check inputs, and unpack

The config cell below runs **after** the mount cell. It checks that all three zips are in My Drive root and creates the output folder. The code zip is then unpacked into `/content/v3`; the two data zips go into separate subfolders so the relative audio paths in the CSVs resolve.

In [ ]:
DRIVE = '/content/drive/MyDrive'
ROOT = '/content/v3'
OUT = f'{DRIVE}/anti_spoof_v3'
NAME = 'xlsr_v3'
EPOCHS = 12
BATCH = 8
WORKERS = 2

import os
os.makedirs(OUT, exist_ok=True)
print('output dir:', OUT)

REQUIRED = ['colab_pack.zip', 'colab_pack_v2.zip', 'colab_v3_code.zip']
missing = []
for f in REQUIRED:
    p = f'{DRIVE}/{f}'
    if os.path.exists(p):
        print(f'  {f:22s} {os.path.getsize(p)/1e6:7.0f} MB')
    else:
        print(f'  {f:22s} MISSING')
        missing.append(f)
if missing:
    raise FileNotFoundError(f'Upload to My Drive root ({DRIVE}): {missing}')

In [ ]:
!rm -rf /content/v3
!mkdir -p /content/v3/colab_pack /content/v3/colab_pack_v2
!unzip -q "$DRIVE/colab_pack.zip"    -d /content/v3/colab_pack
!unzip -q "$DRIVE/colab_pack_v2.zip" -d /content/v3/colab_pack_v2
!unzip -q "$DRIVE/colab_v3_code.zip" -d /content/v3
%cd /content/v3
!ls

In [ ]:
!pip -q install -U "transformers>=4.38,<5" scipy soundfile
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, round(p.total_memory / 1e9, 1), "GB")

## 2. Build the split CSVs

This holds out MLAAD TTS systems for dev, hides the unseen-system test set from train/dev, and writes separate ASV/MLAAD dev files for checkpoint selection.

In [ ]:
!python data_v3.py --root colab_pack --root colab_pack_v2

In [ ]:
import pandas as pd
for f in ["data_v3/v3_train.csv", "data_v3/v3_dev_asv.csv", "data_v3/v3_dev_mlaad.csv",
          "data_v3/v3_test_asv.csv", "data_v3/v3_test_mlaad_unseen.csv"]:
    d = pd.read_csv(f)
    print(f, len(d), d.groupby(["source", "label"]).size().to_dict())

## 3. Sanity check without a GPU model

A tiny randomly-initialised Wav2Vec2 exercises augmentation, dataset, forward/backward, calibration, and split leakage in under a minute.

In [ ]:
!python selftest_v3.py

## 4. Shortcut control (run this before trusting any score)

The MLAAD real/fake clips come from different recording pipelines, so a classifier can separate them from recording style alone. The single-feature AUC below is the number the v3 model has to beat for a real reason. After v3 augmentation this should drop compared with the raw data, but it may not reach 0.5.

In [ ]:
!python evaluate_v3.py --shortcut-only \
  --shortcut-train data_v3/v3_train.csv \
  --shortcut-test data_v3/v3_test_mlaad_unseen.csv \
  --shortcut-n 300 --out "$OUT"

## 5. Train v3

Defaults: all-layer fusion, focal loss, one frozen-SSL warm-up epoch, 12 epochs, early stopping (patience 4), checkpoint chosen by mean AUC over the two dev sets.

The best checkpoint is written to `OUT/xlsr_v3.pt` (~1.2 GB). Keep the tab open; Colab disconnects idle sessions.

In [ ]:
!python train_v3.py \
  --train-csv data_v3/v3_train.csv \
  --dev-csv asv_dev=data_v3/v3_dev_asv.csv \
  --dev-csv mlaad_dev=data_v3/v3_dev_mlaad.csv \
  --name "$NAME" --out "$OUT" \
  --epochs "$EPOCHS" --batch "$BATCH" --workers "$WORKERS" \
  --layers all --loss focal --freeze-ssl-epochs 1 --patience 4 --save-last

### Resume after a disconnect

`--save-last` writes `xlsr_v3_last.pt` every epoch (and `xlsr_v3.pt` only when the dev score improves). `--resume` prefers the last checkpoint, so it continues from the next epoch instead of replaying epochs after the best one. Resuming loads weights (not Adam moments unless the run used `--save-optimizer`), so the first resumed epoch can be slightly noisy. Drive usage is about 2.4 GB (best + last).

In [ ]:
!python train_v3.py \
  --train-csv data_v3/v3_train.csv \
  --dev-csv asv_dev=data_v3/v3_dev_asv.csv \
  --dev-csv mlaad_dev=data_v3/v3_dev_mlaad.csv \
  --name "$NAME" --out "$OUT" \
  --epochs "$EPOCHS" --batch "$BATCH" --workers "$WORKERS" \
  --layers all --loss focal --freeze-ssl-epochs 1 --patience 4 --save-last --resume

## 6. Evaluate on in-domain and cross-domain sets

Calibration is fitted on the dev mix (ASV dev + held-out MLAAD systems), which is disjoint from every test set. The script prints EER, AUC, balanced accuracy, real/fake accuracy, uncertain rate, per-attack results, and saves score CSVs for further analysis.

In [ ]:
!python evaluate_v3.py \
  --ckpt "$OUT/$NAME.pt" \
  --calibrate-csv data_v3/v3_dev.csv \
  --csv asv_eval=data_v3/v3_test_asv.csv \
  --csv mlaad_unseen=data_v3/v3_test_mlaad_unseen.csv \
  --csv mlaad_all=data_v3/v3_test_mlaad_all.csv \
  --csv mlaad_matched=data_v3/v3_test_mlaad_matched.csv \
  --batch 16 --workers "$WORKERS" --out "$OUT/eval"

In [ ]:
import pandas as pd
df = pd.read_csv(f"{OUT}/eval/eval_{NAME}.csv")
cols = ["set", "n", "EER", "AUC", "balanced_acc", "real_acc", "fake_acc", "uncertain_rate"]
print(df[cols].to_string(index=False))

## 7. How to read the result honestly

- **ASVspoof eval EER** should stay near the v1 level (about 1%). A large jump means the multi-domain training hurt in-domain performance.
- **MLAAD unseen-system EER** is the important cross-domain number. v1 was about 46% (chance). Any clear drop below that, with AUC above about 0.6, is real progress on unseen TTS systems.
- **Compare against the shortcut AUC** from section 4. If the model's MLAAD AUC is close to the noise-floor AUC, it may still be using recording style.
- **MLAAD matched** is a small selected subset; treat it as a rough check, not a headline number.
- One seed, one run. Differences of a few points are noise.

If v3 still lands near 45-50% EER on unseen systems, the honest conclusion is that this data/pipeline is not enough for modern TTS, not that the code is broken. Record the numbers and move to a larger, more diverse corpus rather than tuning on the test set.

## 8. Use the calibrated model in the local demo

Copy these two files from Drive to your Mac:

- `anti_spoof_v3/xlsr_v3.pt`
- `anti_spoof_v3/eval/calibration_xlsr_v3.json`

Then, in the project folder:

```bash
cp xlsr_v3.pt checkpoints/
CKPT=checkpoints/xlsr_v3.pt CALIB=eval/calibration_xlsr_v3.json \
  python3 -m uvicorn app:app --port 8000
```

The app reports `real`, `fake`, or `uncertain`. Do not lower the fake threshold just to make your own samples pass; the calibration set is the honest reference.